In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_25_try_0.pickle

In [ ]:
%%RecordEvent
%%time
### cell 26 ###

# 1) Subset extraction: filter columns by question, rename by suffix, drop all‐NaN rows
def grab_subset_of_data_38(original_df, question_of_interest):
    subset = original_df.filter(like=question_of_interest, axis=1)
    if subset.empty:
        return subset
    # keep only the suffix after '-' and strip leading spaces
    new_names = subset.columns.str.split('-').str[-1].str.lstrip()
    subset.columns = new_names
    return subset.dropna(how='all')

# 2) Combine multiple years: vectorized assign of year, concat, count

def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_38(
        question_of_interest, include_2017=None):
    # list of (year, DataFrame) in descending year order
    year_sources = [
        ('2022', responses_df_2022_cell10),
        ('2021', responses_df_2021),
        ('2020', responses_df_2020),
        ('2019', responses_df_2019_cell10),
        ('2018', responses_df_2018_cell10),
    ]
    if include_2017 is not None:
        year_sources.insert(0, ('2017', responses_df_2017))
    # grab subset for each year and tag
    dfs = [
        grab_subset_of_data_38(df, question_of_interest).assign(year=yr)
        for yr, df in year_sources
    ]
    # drop any empty frames
    dfs = [d for d in dfs if not d.empty]
    if not dfs:
        return pd.DataFrame(), pd.DataFrame()
    combined = pd.concat(dfs, ignore_index=True)
    counts = combined.groupby('year').count().reset_index()
    return combined, counts

# 3) Convert counts to percentages by dividing by respondent total per year
def convert_df_of_counts_to_percentages_38(df, df_counts):
    totals = df['year'].value_counts()
    pct = (
        df_counts
        .set_index('year')
        .div(totals, axis=0)
        .mul(100)
        .reset_index()
    )
    return pct

# 4) Normalize column names in the 2018 DataFrame in one chained operation
correct_phrasing = 'Jupyter Notebook / JupyterLab'
incorrect_phrasing = 'Jupyter/IPython'
alternate_phrasing = (
    "Which of the following integrated development environments (IDE's) have you used at work or school in the last 5 years?"
)
question_of_interest_cell38 = (
    "Which of the following integrated development environments (IDE's) do you use on a regular basis?"
)
responses_df_2018_cell10.columns = (
    responses_df_2018_cell10.columns
        .str.replace(incorrect_phrasing, correct_phrasing, regex=False)
        .str.replace(alternate_phrasing, question_of_interest_cell38, regex=False)
)

# 5) Preserve x_axis_title exactly as in original
x_axis_title = 'Percentage of respondents'

# 6) Combine subsets across years and get counts
ide_df_combined, ide_df_combined_counts = (
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_38(
        question_of_interest_cell38
    )
)

# 7) Define mapping of variant columns → unified choice columns
mapping = {
    'Jupyter Notebook / JupyterLab': [
        'Jupyter Notebook', 'JupyterLab ',
        'Jupyter (JupyterLab, Jupyter Notebooks, etc) ',
        'Jupyter Notebook / JupyterLab'
    ],
    'Visual Studio / Visual Studio Code (VSCode)': [
        'Visual Studio Code', 'Visual Studio',
        'Visual Studio / Visual Studio Code ', 'Visual Studio ',
        'Visual Studio Code (VSCode) ', 'Click to write Choice 13'
    ],
    'RStudio': ['RStudio', 'RStudio '],
    'PyCharm': ['PyCharm ', 'PyCharm'],
    'MATLAB': ['MATLAB', 'MATLAB '],
}

# 8) Build cleaned DataFrame: one column per choice, mark True→choice else NaN
ide_df_clean = ide_df_combined.copy()
for choice, variants in mapping.items():
    mask = ide_df_clean[variants].notna().any(axis=1)
    ide_df_clean[choice] = choice
    ide_df_clean.loc[~mask, choice] = np.nan
# drop all raw variant columns
raw_cols = [v for vs in mapping.values() for v in vs]
ide_df_clean.drop(columns=raw_cols, inplace=True)

# 9) Recompute counts and percentages for cleaned data
ide_df_counts2 = ide_df_clean.groupby('year').count().reset_index()
ide_df_percentages = convert_df_of_counts_to_percentages_38(ide_df_clean, ide_df_counts2)

# 10) Select and order the five choices for plotting
cols_order = [
    'Jupyter Notebook / JupyterLab',
    'Visual Studio / Visual Studio Code (VSCode)',
    'RStudio',
    'PyCharm',
    'MATLAB'
]
ide_df_percentages_cell38 = ide_df_percentages[['year'] + cols_order]

# 11) Melt into long form, sort, and reset index

df_cell38 = (
    ide_df_percentages_cell38
        .melt(id_vars=['year'], value_vars=cols_order, var_name='', value_name='value')
        .sort_values(['year', 'value'])
        .reset_index(drop=True)
)

# 12) Confirm no unintended modifications
x_axis_title  # should still be 'Percentage of respondents'
df_cell38.info()

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_26_try_1.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_26_try_1.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[26], f)


In [ ]:
opt_output = Out.get(4)